In [8]:
!pip install category_encoders

zsh:1: command not found: pip


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import csv
import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from skimage import color, data, img_as_float
from skimage.transform import resize, rotate

print("Libraries imported successfully.")

Libraries imported successfully.


In [69]:
dataset = pd.read_csv('NOLA_Surveillance.csv')
dataset.head()

/var/folders/3p/wpdcwlx57k93mldysxqtnprw0000gn/T/ipykernel_95985/3020394071.py:1: DtypeWarning: Columns (17,20,21,34) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv('NOLA_Surveillance.csv')


,agency_code,agency_collection_num,collection_id,code,name,street,city,zip,region,site_code,...,disease_week,num_trap,trap_nights,trap_problem,comments,species,add_date,add_user,males,females - mixed
0,NOLA,1,2998124,504042,778 HARRISON AVE,778 HARRISON AVE,New Orleans,70124.0,"Orleans County (Louisiana, USA)",504042,...,9,1,1,N,NaN,Culex quinquefasciatus,2023-09-21 14:35,NaN,NaN,11.0
1,NOLA,2,2998125,504046,987 ROBERT E LEE BLVD,987 Allen Toussaint Blvd,New Orleans,70124.0,"Orleans County (Louisiana, USA)",504046,...,9,1,1,N,NaN,Culex quinquefasciatus,2023-09-21 14:35,NaN,NaN,7.0
2,NOLA,3,2998126,504025,401 CITY PARK AVE,401 CITY PARK AVE,New Orleans,70119.0,"Orleans County (Louisiana, USA)",504025,...,9,1,1,N,NaN,Culex quinquefasciatus,2023-09-21 14:35,NaN,NaN,3.0
3,NOLA,4,2998127,504045,964 N CARROLLTON AVE,964 N CARROLLTON AVE,New Orleans,70119.0,"Orleans County (Louisiana, USA)",504045,...,9,1,1,N,NaN,Culex quinquefasciatus,2023-09-21 14:35,NaN,3.0,16.0
4,NOLA,5,2998128,504002,1001 HARRISON AVE,1001 HARRISON AVE,New Orleans,70124.0,"Orleans County (Louisiana, USA)",504002,...,9,1,1,N,NaN,Culex quinquefasciatus,2023-09-21 14:35,NaN,NaN,12.0


In [70]:
#How many rows and columns does the dataset have?
dataset.shape 

(71697, 40)

In [71]:
dataset.isnull().sum() #shows the missing data

agency_code                             0
agency_collection_num                   0
collection_id                           0
code                                    0
name                                    0
street                              14695
city                                17863
zip                                 21096
region                                  0
site_code                               0
site_name                               0
site_street                         14695
site_city                           20456
site_zip                            21096
site_region                             0
calculated_neighborhood             71697
calculated_neighborhood_distance    71697
calculated_city                     66781
calculated_city_distance            66781
calculated_subcounty                    0
calculated_county                   66781
calculated_state                    66781
longitude                               0
latitude                          

In [84]:
#filter through columns and rows
file_path = 'NOLA_Surveillance.csv'

#columns = pd.read_csv(filepath, nrows=0).columns.tolist()


columns_to_keep = [
    "trap_type",
    "collection_date",
    "species",
    "females - mixed",
]
df = pd.read_csv(file_path, usecols=columns_to_keep)

species_matches = (
    (df["species"]
    .str.strip()
    .str.casefold()
    .eq("culex quinquefasciatus")
)
)
filtered_df = df.loc[species_matches].copy()

In [73]:
print("Number of rows:", len(filtered_df))#display how many rows and columns are shown

Number of rows: 22813


In [85]:
filtered_df.isnull().sum() #check missing or null values in your data

trap_type            0
collection_date      0
species              0
females - mixed    224
dtype: int64

In [86]:
filtered_df["collection_date"] = pd.to_datetime(
    filtered_df["collection_date"],
    errors="coerce"
)

filtered_df["week_monday"] = (
    filtered_df["collection_date"]
    - pd.to_timedelta(
        filtered_df["collection_date"].dt.weekday,
        unit="D"
    )
)

In [87]:
#this find the average female count of each trap for one day

def calculate_total(values): #this function adds the female mosquito counts 
    return values.sum(min_count=1)

weekly_summary = (
filtered_df
.groupby('week_monday')
.agg(
    total_females=("females - mixed", calculate_total),
    number_of_traps=("females - mixed", "size")
)
.reset_index()
)
weekly_summary ["females - mixed"] = (
    daily_summary["total_females"] /
    daily_summary["number_of_traps"]
)


display(weekly_summary)

,week_monday,total_females,number_of_traps,females - mixed
0,2013-01-07,426.0,16,26.625000
1,2013-01-14,11.0,8,1.375000
2,2013-01-21,72.0,10,7.200000
3,2013-01-28,398.0,11,36.181818
4,2013-02-04,236.0,12,19.666667
...,...,...,...,...
498,2024-11-11,2331.0,48,2.000000
499,2024-11-18,647.0,41,9.800000
500,2024-11-25,NaN,1,29.772727
501,2024-12-02,349.0,44,1.333333


In [88]:
output_df = (weekly_summary[["week_monday", "females - mixed"]]
    .rename(columns={"week_monday": "date", "females - mixed": "mosq_per_trap_night"
                     }
                     )
.copy()
)

display(output_df)

output_df.to_csv(
    "NOLA_culexquin.csv",
    index=False
)

print("NOLA_culexquin.csv is created")

,date,mosq_per_trap_night
0,2013-01-07,26.625000
1,2013-01-14,1.375000
2,2013-01-21,7.200000
3,2013-01-28,36.181818
4,2013-02-04,19.666667
...,...,...
498,2024-11-11,2.000000
499,2024-11-18,9.800000
500,2024-11-25,29.772727
501,2024-12-02,1.333333


NOLA_culexquin.csv is created
